# Week 3 — Advanced RAG + Neo4j Knowledge Graph

**What's different:**
- Vector RAG with proper chunking strategies (fixed vs semantic vs overlap)
- **Neo4j Graph RAG** — model IT assets, incidents, and CMDB relationships as a graph
- Hybrid retrieval: vector similarity + Cypher graph traversal
- RAGAS-style eval: faithfulness + answer relevance scored automatically
- Full observability: token/cost/latency logged per retrieval call
- Chatbot: source citations + Neo4j "related issues" panel

**APIs:** OpenAI embeddings + `gpt-4o-mini`, Open-Meteo (no auth), Wikipedia (no auth), Neo4j

**Neo4j setup:**
```bash
# Option A: Neo4j Desktop (free)
# Download from https://neo4j.com/download/
# Option B: Neo4j Aura Free (cloud, no install)
# https://neo4j.com/cloud/aura-free/
pip install neo4j
```

---
### Agenda
1. Setup + Neo4j connection
2. Knowledge base — realistic IT ops docs
3. Chunking strategies — why chunk design matters
4. Vector RAG — the full pipeline with observability
5. Neo4j CMDB schema — nodes, relationships, Cypher
6. Graph RAG — hybrid vector + graph retrieval
7. RAG failure modes + mitigations
8. RAGAS-style evaluation
9. Chatbot: citations + related issues panel

In [1]:
!pip install openai anthropic neo4j scikit-learn numpy requests pydantic --quiet




[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\vivik\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
mport sys, os, json, requests, numpy as np
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from shared.utils import (
    LLMClient, Tracer, EvalFramework, Guardrails,
    banner, section, observe, discuss, compare
)
from sklearn.metrics.pairwise import cosine_similarity

# ── Neo4j connection ──────────────────────────────────────────────────────────
# Set these env vars or paste your Aura/Desktop credentials
NEO4J_URI      = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
NEO4J_USER     = os.getenv("NEO4J_USER",     "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")

# Test connection
NEO4J_AVAILABLE = False
try:
    from neo4j import GraphDatabase
    _driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    _driver.verify_connectivity()
    NEO4J_AVAILABLE = True
    print("✅  Neo4j connected")
except Exception as e:
    print(f"⚠️   Neo4j not available ({e[:80]}). Running in MOCK MODE for graph sections.")

tracer = Tracer(session_id="w03")
client = LLMClient(tracer=tracer)
GPT    = "gpt-4o-mini"
print("✅  Setup complete")

---
## Section 1 — Knowledge Base (Realistic IT Ops Docs)

In [ ]:
# ── 20 realistic IT ops knowledge base documents ──────────────────────────────
KNOWLEDGE_BASE = [
    {"id":"sla-p1",   "title":"SLA: P1 Critical Incidents",
     "content":"P1 incidents: acknowledge within 15 min, resolve within 4 hours. Definition: complete service outage >50 users, revenue-impacting failure, active security breach. Must page PagerDuty on-call immediately. Post-incident review required within 5 business days. P1 automatically opens a war room in #incidents-live Slack channel."},

    {"id":"sla-p2",   "title":"SLA: P2 High Priority",
     "content":"P2 incidents: acknowledge within 30 min, resolve within 8 business hours. Definition: single team blocked, critical user unable to work, degraded performance on key system. SEV2 in PagerDuty. SLA clock pauses in 'Awaiting User' state."},

    {"id":"sla-p3p4", "title":"SLA: P3 and P4 Incidents",
     "content":"P3: acknowledge 2h, resolve 3 business days. P4: acknowledge 1 day, resolve 10 business days. P3 examples: individual software issue with workaround, non-urgent access request. P4 examples: cosmetic UI changes, informational queries. SLA clock stops during 'Awaiting User'."},

    {"id":"vpn-current","title":"VPN Policy — GlobalProtect (Current)",
     "content":"All remote access uses GlobalProtect VPN (Palo Alto). Cisco AnyConnect decommissioned March 2024. MFA mandatory via Okta Verify. Split tunnelling disabled — all traffic routes through corporate. Max session: 12 hours. Install: intranet.company.com/vpn-install. TAP adapter errors: reinstall the client, not just the driver."},

    {"id":"vpn-legacy","title":"VPN Policy — Cisco AnyConnect (DEPRECATED March 2024)",
     "content":"DEPRECATED. Do not follow. Cisco AnyConnect was decommissioned 31 March 2024. See vpn-current for current policy. Legacy doc kept for audit trail only."},

    {"id":"dw-access", "title":"Data Warehouse Access Request (Snowflake)",
     "content":"Submit Data Access Request via IT Service Portal (portal.internal/data-access). Required: business justification, data classification (Public/Internal/Confidential/Restricted), duration, manager approval. Confidential/Restricted datasets also need DPO sign-off. Standard: 3 business days. Urgent (business case required): 24 hours. All access logged, reviewed quarterly."},

    {"id":"pii-cloud", "title":"PII Storage Policy — Cloud Object Storage",
     "content":"PII in AWS S3/GCS/Azure Blob permitted ONLY with: (1) server-side encryption AES-256+, (2) public access blocked at account level, (3) tagged 'PII-Restricted', (4) retention not exceeding purpose (GDPR Art. 5), (5) access via IAM roles only — no static credentials. Violations must be reported to Security within 24 hours of discovery."},

    {"id":"oncall-sap", "title":"After-Hours SAP Support",
     "content":"SAP Basis issues outside 8 AM–7 PM IST or weekends: (1) Page 'SAP-BASIS-ONCALL' PagerDuty schedule. (2) No response in 15 min → escalate to SAP CoE Lead Anand Krishnamurthy. (3) SAP HANA DB specifically → 'HANA-DBA-ONCALL'. Business hours: raise ServiceNow ticket, category SAP, assignment group SAP-BASIS-TEAM. Maintenance windows: Saturday 11 PM – 3 AM IST."},

    {"id":"change-mgmt","title":"Change Management — CAB and Emergency Changes",
     "content":"Standard changes: submit ServiceNow, reviewed at CAB Thursdays 2 PM IST. Needs: manager approval + IT Risk for production changes. Emergency changes: bypass CAB, requires Emergency Change Manager approval (24/7 via PagerDuty). All changes: rollback plan + test evidence + business impact statement. Change freeze: Dec 15–Jan 5, plus 2 weeks around major releases."},

    {"id":"backup",    "title":"Backup and Disaster Recovery",
     "content":"Production DBs: daily full + continuous WAL. RPO=1h, RTO=4h. File servers: daily incremental + weekly full. RPO=24h, RTO=8h. Laptops: Backblaze cloud backup on corporate Wi-Fi. Retention: 30 days rolling + 12 months monthly snapshot. DR test: quarterly. Last test Q3 2024, pass rate 94%. Restore request: P2 ServiceNow ticket, category Data Recovery."},

    {"id":"incident-major","title":"Major Incident (SEV1) Response Process",
     "content":"SEV1 = P1. Triggers Major Incident Response: (1) Incident Commander from on-call rotation. (2) War room in #incidents-live within 5 min. (3) status.company.com updated within 10 min. (4) exec-alerts distro notified. (5) Customer comms team if >10 customers affected. Resolution: all primary services restored AND monitoring green for 15 min."},

    {"id":"cmdb-policy","title":"CMDB Governance — Configuration Item Standards",
     "content":"All production CIs must be registered in ServiceNow CMDB within 24 hours of provisioning. Required attributes: CI name, type, owner team, environment (prod/staging/dev), upstream/downstream dependencies. CMDB accuracy target: 95%. Quarterly CMDB audits run by IT Risk. Unregistered CIs found in production: raise a P3 change record."},

    {"id":"gdpr-sar",  "title":"GDPR Subject Access Request Handling",
     "content":"SARs must be fulfilled within 30 days (GDPR Art. 15). IT must respond to data extraction requests from the DPO within 5 business days. GDPR reporting dashboard: reports.internal/gdpr-sar. If dashboard is unavailable, escalate to IT Security as P2 — regulatory deadline risk. Archive SAR evidence for 3 years."},
]

print(f"Knowledge base: {len(KNOWLEDGE_BASE)} documents loaded")
for doc in KNOWLEDGE_BASE:
    print(f"  [{doc['id']}] {doc['title']}")

---
## Section 2 — Chunking Strategies

In [ ]:
# ── Chunking strategies comparison ────────────────────────────────────────────
banner("Chunking strategies — why design matters")

def chunk_fixed(text: str, chunk_size: int = 200, overlap: int = 0) -> list:
    words = text.split()
    chunks, i = [], 0
    while i < len(words):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

def chunk_by_sentence(text: str, max_sentences: int = 3) -> list:
    import re
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks, i = [], 0
    while i < len(sentences):
        chunks.append(" ".join(sentences[i:i+max_sentences]))
        i += max_sentences
    return chunks

# Test on the backup policy
backup_text = KNOWLEDGE_BASE[9]["content"]

section("Fixed chunking (no overlap) — 50 words")
fixed_no_overlap = chunk_fixed(backup_text, chunk_size=50, overlap=0)
for i, c in enumerate(fixed_no_overlap):
    print(f"  Chunk {i+1}: {c[:80]}...")

print()
section("Fixed chunking WITH 15-word overlap")
fixed_overlap = chunk_fixed(backup_text, chunk_size=50, overlap=15)
for i, c in enumerate(fixed_overlap):
    print(f"  Chunk {i+1}: {c[:80]}...")

print()
observe("Overlap ensures a key fact at the end of chunk N also appears at the start of chunk N+1.\n"
        "  Without overlap, the answer to 'What is the RPO?' can be split across chunks.")

---
## Section 3 — Vector RAG Pipeline with Full Observability

In [ ]:
# ── KnowledgeBase class — production-grade vector store ───────────────────────
class KnowledgeBase:
    """
    Production-grade vector knowledge base.
    - Embeds documents with metadata
    - Retrieves with score threshold + query expansion
    - Logs every retrieval call through the tracer
    """

    def __init__(self, client: LLMClient, embed_model: str = "text-embedding-3-small"):
        self.client      = client
        self.embed_model = embed_model
        self.docs        = []
        self.embeddings  = None

    def add_documents(self, docs: list) -> None:
        """Add documents and embed them."""
        self.docs = docs
        texts = [f"{d['title']}\n{d['content']}" for d in docs]
        print(f"Embedding {len(texts)} documents...")
        self.embeddings = self.client.embed(texts, model=self.embed_model)
        print(f"Done. Embedding shape: {self.embeddings.shape}")

    def retrieve(
        self,
        query: str,
        top_k: int = 3,
        score_threshold: float = 0.25,
        expand_query: bool = False
    ) -> list:
        """Retrieve top_k most relevant documents."""
        queries = [query]

        if expand_query:
            expansion = self.client.chat(
                GPT,
                user=f"Generate 3 alternate phrasings of this query. One per line. No numbering.\nQuery: {query}",
                temperature=0.4, tags=["query_expansion"]
            )
            queries += [q.strip() for q in expansion.strip().splitlines() if q.strip()]

        # Embed all query variants, aggregate max score per doc
        best_scores = np.zeros(len(self.docs))
        for q in queries:
            q_emb = self.client.embed(q, model=self.embed_model)
            scores = cosine_similarity(q_emb, self.embeddings)[0]
            best_scores = np.maximum(best_scores, scores)

        ranked = np.argsort(best_scores)[::-1]
        results = []
        for idx in ranked[:top_k]:
            if best_scores[idx] >= score_threshold:
                results.append({
                    "doc":   self.docs[idx],
                    "score": float(best_scores[idx]),
                    "rank":  len(results) + 1
                })
        return results

    def answer(
        self,
        question: str,
        top_k: int = 3,
        expand_query: bool = False,
        verbose: bool = True
    ) -> dict:
        """
        Full RAG pipeline: retrieve → augment → generate.
        Returns {answer, sources, retrieved_count, used_expansion}
        """
        retrieved = self.retrieve(question, top_k=top_k, expand_query=expand_query)

        if verbose:
            print(f"📎 Retrieved {len(retrieved)} docs (expand={expand_query}):")
            for r in retrieved:
                print(f"   [{r['doc']['id']}] score={r['score']:.3f}  {r['doc']['title']}")
            print()

        if not retrieved:
            return {
                "answer":   "I don't have information about this in the knowledge base.",
                "sources":  [],
                "retrieved_count": 0,
                "used_expansion": expand_query
            }

        context = "\n\n".join([
            f"[SOURCE: {r['doc']['id']}] {r['doc']['title']}\n{r['doc']['content']}"
            for r in retrieved
        ])

        RAG_SYSTEM = """
You are an IT knowledge base assistant.
Answer using ONLY the provided context. 
Cite sources as [doc-id] after each claim.
If context has conflicting information (e.g., old and new policy), flag the conflict.
If the answer is not in the context, say: "Not covered in the knowledge base."
Never guess or extrapolate beyond the context.
"""
        answer = self.client.chat(
            GPT,
            user=f"CONTEXT:\n{context}\n\nQUESTION: {question}",
            system=RAG_SYSTEM,
            temperature=0,
            tags=["rag", "answer"]
        )

        return {
            "answer":   answer,
            "sources":  [r["doc"]["id"] for r in retrieved],
            "retrieved_count": len(retrieved),
            "used_expansion": expand_query
        }

# ── Build the KB ───────────────────────────────────────────────────────────────
kb = KnowledgeBase(client)
kb.add_documents(KNOWLEDGE_BASE)

In [ ]:
# ── RAG happy path ────────────────────────────────────────────────────────────
banner("RAG — happy path questions")

questions = [
    "What is our SLA for a P1 incident?",
    "Can I store customer email addresses in an S3 bucket?",
    "How do I request urgent access to the data warehouse?",
]

for q in questions:
    print(f"Q: {q}")
    result = kb.answer(q, verbose=True)
    print(f"A: {result['answer'].strip()[:400]}")
    print(f"   Sources: {result['sources']}")
    print()

In [ ]:
# ── RAG failure modes ────────────────────────────────────────────────────────
banner("RAG Failure Mode 1 — Vocabulary mismatch (SEV1 vs P1)")

q_p1   = "What is the SLA for a P1 incident?"
q_sev1 = "What is the SLA for a SEV1 incident?"

r_p1   = kb.retrieve(q_p1,   top_k=2)
r_sev1 = kb.retrieve(q_sev1, top_k=2)

print(f"'P1' retrieves:   {[r['doc']['id'] for r in r_p1]}  scores={[f'{r["score"]:.3f}' for r in r_p1]}")
print(f"'SEV1' retrieves: {[r['doc']['id'] for r in r_sev1]} scores={[f'{r["score"]:.3f}' for r in r_sev1]}")
print()
print("With query expansion:")
r_sev1_expanded = kb.retrieve(q_sev1, top_k=2, expand_query=True)
print(f"  Retrieves: {[r['doc']['id'] for r in r_sev1_expanded]}")

observe("Our KB has sla-p1 (uses 'P1') and incident-major (uses 'SEV1') — same concept, different terms.\n"
        "  Query expansion bridges this gap. Without it, 'SEV1' queries miss the SLA doc.")

In [ ]:
# ── Failure Mode 2: Conflicting documents (VPN) ───────────────────────────────
banner("RAG Failure Mode 2 — Conflicting old and new policy")

vpn_q = "How do I connect to the company VPN?"
result = kb.answer(vpn_q, top_k=3, verbose=True)
print(f"Answer: {result['answer'].strip()[:500]}")
print(f"Sources: {result['sources']}")
print()
if "vpn-legacy" in result["sources"]:
    print("⚠️  Retrieved deprecated policy (vpn-legacy). Did the model flag the conflict?")

observe("Fix: tag deprecated docs in metadata and filter them before retrieval.\n"
        "  Or add 'DEPRECATED' to the title — the embedding will score it lower for direct queries.")

---
## Section 4 — Neo4j CMDB Schema + Graph RAG

In [ ]:
# ── Neo4j schema setup ────────────────────────────────────────────────────────
banner("Neo4j CMDB — nodes, relationships, Cypher setup")

CMDB_SETUP_CYPHER = """
// ── Nodes ──────────────────────────────────────────────────────────────────
// ConfigItem (CI)
CREATE CONSTRAINT ci_id IF NOT EXISTS FOR (c:ConfigItem) REQUIRE c.id IS UNIQUE;

// Incident
CREATE CONSTRAINT inc_id IF NOT EXISTS FOR (i:Incident) REQUIRE i.number IS UNIQUE;

// Team
CREATE CONSTRAINT team_name IF NOT EXISTS FOR (t:Team) REQUIRE t.name IS UNIQUE;

// ── Sample CMDB data ─────────────────────────────────────────────────────────
MERGE (sap:ConfigItem {id:'CI-SAP-001', name:'SAP S/4HANA', type:'Application',
       environment:'Production', criticality:'P1'})
MERGE (hana:ConfigItem {id:'CI-HANA-001', name:'SAP HANA DB', type:'Database',
       environment:'Production', criticality:'P1'})
MERGE (lb:ConfigItem  {id:'CI-LB-001',  name:'Load Balancer (F5)',  type:'Network',
       environment:'Production', criticality:'P1'})
MERGE (vpn:ConfigItem {id:'CI-VPN-001', name:'GlobalProtect VPN',   type:'Network',
       environment:'Production', criticality:'P2'})
MERGE (snow:ConfigItem {id:'CI-SN-001', name:'ServiceNow Platform',  type:'Application',
       environment:'Production', criticality:'P1'})
MERGE (dw:ConfigItem  {id:'CI-DW-001',  name:'Snowflake Data Warehouse', type:'Database',
       environment:'Production', criticality:'P2'})

// ── Teams ────────────────────────────────────────────────────────────────────
MERGE (sap_team:Team {name:'SAP-BASIS-TEAM'})
MERGE (net_team:Team {name:'Network-Ops'})
MERGE (app_team:Team {name:'App-Support'})
MERGE (dba_team:Team {name:'DBA-Team'})

// ── Relationships ─────────────────────────────────────────────────────────────
MERGE (sap)-[:DEPENDS_ON]->(hana)
MERGE (sap)-[:DEPENDS_ON]->(lb)
MERGE (snow)-[:DEPENDS_ON]->(lb)
MERGE (sap)-[:OWNED_BY]->(sap_team)
MERGE (hana)-[:OWNED_BY]->(dba_team)
MERGE (lb)-[:OWNED_BY]->(net_team)
MERGE (vpn)-[:OWNED_BY]->(net_team)
MERGE (dw)-[:OWNED_BY]->(dba_team)

// ── Sample incidents linked to CIs ───────────────────────────────────────────
MERGE (inc1:Incident {number:'INC0001001', description:'SAP login slow for all users',
       priority:'P2', status:'Resolved', root_cause:'HANA memory pressure'})
MERGE (inc2:Incident {number:'INC0001002', description:'VPN TAP adapter missing after Windows update',
       priority:'P2', status:'Resolved', root_cause:'Driver conflict'})
MERGE (inc3:Incident {number:'INC0001003', description:'Load balancer SSL cert expired',
       priority:'P1', status:'Resolved', root_cause:'Certificate not monitored'})
MERGE (inc4:Incident {number:'INC0001004', description:'ServiceNow portal 503 errors',
       priority:'P1', status:'Resolved', root_cause:'Load balancer SSL cert expired'})

// Incidents affect CIs
MERGE (inc1)-[:AFFECTS]->(sap)
MERGE (inc1)-[:AFFECTS]->(hana)
MERGE (inc2)-[:AFFECTS]->(vpn)
MERGE (inc3)-[:AFFECTS]->(lb)
MERGE (inc4)-[:AFFECTS]->(snow)
MERGE (inc4)-[:AFFECTS]->(lb)   // Same root cause
"""

if NEO4J_AVAILABLE:
    with _driver.session() as session:
        for stmt in CMDB_SETUP_CYPHER.strip().split(";"):
            stmt = stmt.strip()
            if stmt:
                try:
                    session.run(stmt)
                except Exception as e:
                    print(f"  Warning: {e}")
    print("✅  CMDB schema and sample data created in Neo4j")
else:
    print("[MOCK MODE] Neo4j not available. Showing Cypher queries as documentation.")
    print(CMDB_SETUP_CYPHER[:500], "...")

In [ ]:
# ── Graph queries ──────────────────────────────────────────────────────────────
banner("Graph RAG — Cypher queries for impact analysis")

# Mock data for when Neo4j is not available
MOCK_GRAPH_DATA = {
    "incident_history_sap": [
        {"incident": "INC0001001", "description": "SAP login slow for all users",
         "priority": "P2", "root_cause": "HANA memory pressure"},
    ],
    "impact_analysis_lb": [
        {"ci": "CI-SAP-001", "name": "SAP S/4HANA", "type": "Application"},
        {"ci": "CI-SN-001",  "name": "ServiceNow Platform", "type": "Application"},
    ],
    "related_incidents_lb": [
        {"incident": "INC0001003", "description": "Load balancer SSL cert expired",
         "priority": "P1", "root_cause": "Certificate not monitored"},
        {"incident": "INC0001004", "description": "ServiceNow portal 503 errors",
         "priority": "P1", "root_cause": "Load balancer SSL cert expired"},
    ],
}

def run_cypher(query: str, params: dict = {}) -> list:
    if NEO4J_AVAILABLE:
        with _driver.session() as session:
            result = session.run(query, params)
            return [dict(r) for r in result]
    return []  # Return empty in mock mode; we use MOCK_GRAPH_DATA below


# Query 1: Incident history for a CI
section("Query 1: What incidents has SAP S/4HANA had?")
Q1 = """
MATCH (inc:Incident)-[:AFFECTS]->(ci:ConfigItem {name: $ci_name})
RETURN inc.number AS incident, inc.description AS description,
       inc.priority AS priority, inc.root_cause AS root_cause
ORDER BY inc.priority
"""
rows = run_cypher(Q1, {"ci_name": "SAP S/4HANA"}) or MOCK_GRAPH_DATA["incident_history_sap"]
for r in rows:
    print(f"  [{r.get('incident',r.get('inc','?'))}] {r.get('description','')} — {r.get('root_cause','')}")

# Query 2: Impact analysis — if LB goes down, what else breaks?
section("Query 2: Impact analysis — if Load Balancer fails, what is affected?")
Q2 = """
MATCH (affected:ConfigItem)-[:DEPENDS_ON*1..3]->(lb:ConfigItem {name: 'Load Balancer (F5)'})
RETURN affected.id AS ci, affected.name AS name, affected.type AS type
"""
rows = run_cypher(Q2) or MOCK_GRAPH_DATA["impact_analysis_lb"]
print("  If Load Balancer fails, these CIs are affected:")
for r in rows:
    print(f"  → {r.get('name','?')} ({r.get('type','?')})")

# Query 3: Find related past incidents by root cause
section("Query 3: Are there past incidents with similar root cause to SSL cert expiry?")
Q3 = """
MATCH (inc:Incident)-[:AFFECTS]->(ci:ConfigItem)
WHERE inc.root_cause CONTAINS 'cert' OR inc.root_cause CONTAINS 'SSL'
RETURN inc.number AS incident, inc.description AS description,
       inc.priority AS priority, inc.root_cause AS root_cause
"""
rows = run_cypher(Q3) or MOCK_GRAPH_DATA["related_incidents_lb"]
for r in rows:
    print(f"  [{r.get('incident',r.get('inc','?'))}] P={r.get('priority','?')}: {r.get('description','')}")

observe("Graph RAG answers questions that pure vector search can't:\n"
        "  'What systems break if X goes down?' requires traversing relationships, not just semantic matching.")

In [ ]:
# ── Hybrid retrieval: vector + graph ─────────────────────────────────────────
banner("Hybrid RAG — vector + graph combined")

def hybrid_rag_answer(question: str, ci_name: str = None, verbose: bool = True) -> str:
    """
    Combines:
    - Vector retrieval from KB (policies, procedures)
    - Graph retrieval from CMDB (CI dependencies, past incidents)
    Merges into a single context for the LLM.
    """
    context_parts = []

    # Part 1: Vector KB
    retrieved = kb.retrieve(question, top_k=3, expand_query=True)
    if retrieved:
        kb_ctx = "\n\n".join([
            f"[POLICY: {r['doc']['id']}]\n{r['doc']['content']}"
            for r in retrieved
        ])
        context_parts.append(f"## POLICIES AND PROCEDURES\n{kb_ctx}")

    # Part 2: Graph CMDB (only if a CI name is relevant)
    if ci_name:
        # Incident history
        inc_rows = run_cypher(Q1, {"ci_name": ci_name}) or MOCK_GRAPH_DATA.get("incident_history_sap", [])
        if inc_rows:
            inc_ctx = json.dumps(inc_rows, indent=2)
            context_parts.append(f"## CMDB: PAST INCIDENTS FOR {ci_name}\n{inc_ctx}")

        # Impact analysis
        imp_rows = run_cypher(Q2) or MOCK_GRAPH_DATA.get("impact_analysis_lb", [])
        if imp_rows:
            imp_ctx = json.dumps(imp_rows, indent=2)
            context_parts.append(f"## CMDB: DEPENDENCY IMPACT\n{imp_ctx}")

    if not context_parts:
        return "Not covered in the knowledge base or CMDB."

    full_context = "\n\n".join(context_parts)
    if verbose:
        print(f"Context size: ~{len(full_context.split())} words from {len(context_parts)} sources")

    HYBRID_SYSTEM = """
You are an IT knowledge assistant with access to both policy documents and CMDB data.
Answer using the provided context. Cite sources as [POLICY: id] or [CMDB].
If policies and CMDB conflict, note it. Never guess.
"""
    return client.chat(
        GPT,
        user=f"CONTEXT:\n{full_context}\n\nQUESTION: {question}",
        system=HYBRID_SYSTEM,
        temperature=0, tags=["hybrid_rag"]
    )

# Test hybrid query
q = "SAP is slow again. What is the SLA and has this happened before?"
print(f"Q: {q}\n")
answer = hybrid_rag_answer(q, ci_name="SAP S/4HANA")
print(f"A: {answer.strip()}")

---
## Section 5 — RAGAS-Style Evaluation

In [ ]:
# ── RAGAS-style eval: faithfulness + answer relevance ─────────────────────────
banner("RAGAS-style RAG evaluation")

def eval_faithfulness(question: str, answer: str, context: str) -> dict:
    """
    Faithfulness: is every claim in the answer supported by the context?
    Score: 0.0 (hallucinated) → 1.0 (fully grounded)
    """
    prompt = f"""
Rate the faithfulness of this answer to the given context.
Faithfulness = fraction of answer claims that are supported by the context.

Context: {context[:800]}
Question: {question}
Answer: {answer[:400]}

Return JSON: {{"score": 0.0-1.0, "unsupported_claims": [list of claims not in context], "reason": "brief"}}
"""
    raw = client.chat(GPT, user=prompt, temperature=0, json_mode=True, tags=["eval","faithfulness"])
    try:
        return json.loads(raw)
    except Exception:
        return {"score": 0.0, "reason": "parse error"}

def eval_answer_relevance(question: str, answer: str) -> dict:
    """
    Answer relevance: does the answer actually address the question?
    """
    prompt = f"""
Rate how well this answer addresses the question.
Score: 1.0 = perfectly relevant, 0.0 = completely off-topic.

Question: {question}
Answer: {answer[:400]}

Return JSON: {{"score": 0.0-1.0, "reason": "brief"}}
"""
    raw = client.chat(GPT, user=prompt, temperature=0, json_mode=True, tags=["eval","relevance"])
    try:
        return json.loads(raw)
    except Exception:
        return {"score": 0.0, "reason": "parse error"}


# Run RAG eval on 5 questions
eval_questions = [
    "What is our SLA for a P1 incident and what happens after it's resolved?",
    "Can I store customer phone numbers in S3?",
    "Who is the on-call person for SAP on Saturday night?",
    "What is our maternity leave policy?",   # Out of scope — should say not covered
    "How do I connect to VPN from home?",
]

print(f"{'Question':<55} {'Faith':>6} {'Relev':>6} {'Avg':>6}")
print("─" * 75)

for q in eval_questions:
    retrieved = kb.retrieve(q, top_k=2)
    ctx = " ".join([r["doc"]["content"] for r in retrieved])
    result = kb.answer(q, top_k=2, verbose=False)
    ans = result["answer"]

    faith = eval_faithfulness(q, ans, ctx)
    relev = eval_answer_relevance(q, ans)

    f_score = faith.get("score", 0)
    r_score = relev.get("score", 0)
    avg     = (f_score + r_score) / 2

    flag = "⚠️" if avg < 0.7 else "✅"
    print(f"{flag} {q[:53]:<55} {f_score:>6.2f} {r_score:>6.2f} {avg:>6.2f}")

In [ ]:
# ── Chatbot: RAG + Neo4j related issues panel ─────────────────────────────────
banner("Week 3 Chatbot — citations + related issues panel")

streamlit_w3 = '''
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))
import streamlit as st, json
from shared.utils import LLMClient, Tracer
# (KnowledgeBase would be imported from w03 in a real setup)

st.set_page_config(page_title="RAG Helpdesk — W3", page_icon="📚")
st.title("📚  IT Knowledge Base Chatbot")
st.caption("Week 3 — RAG with source citations")

if "history" not in st.session_state: st.session_state.history = []
tracer = Tracer("streamlit-w03")
client = LLMClient(tracer=tracer)

col1, col2 = st.columns([2, 1])

with col1:
    st.subheader("Chat")
    for msg in st.session_state.history:
        with st.chat_message(msg["role"]): st.write(msg["content"])
    if prompt := st.chat_input("Ask about IT policies..."):
        st.session_state.history.append({"role":"user","content":prompt})
        # In production: call kb.answer(prompt)
        # For demo: use direct LLM with sample context
        response = client.chat("gpt-4o-mini", user=prompt, temperature=0,
                               system="You are an IT policy assistant. Be helpful.",
                               tags=["streamlit"])
        st.session_state.history.append({"role":"assistant","content":response})
        st.rerun()

with col2:
    st.subheader("Session Stats")
    st.metric("Total cost", f"${tracer.total_cost():.5f}")
    st.metric("Calls", len(tracer.traces))
    st.subheader("Related Issues")
    st.info("In production: Neo4j query would show related past incidents here.")
    if st.button("Clear"): st.session_state.history=[]; st.rerun()
'''

with open("/tmp/chatbot_w03.py", "w") as f:
    f.write(streamlit_w3.strip())
print("Saved: /tmp/chatbot_w03.py")
print("Run: streamlit run /tmp/chatbot_w03.py")

In [ ]:
banner("Session summary")
tracer.summary()

---
## Week 3 — Key Takeaways

| Concept | What we built | Production implication |
|---|---|---|
| Chunking | Fixed vs sentence vs overlap | Wrong chunk size = split answers = hallucination |
| Query expansion | SEV1 → retrieves P1 docs | Always expand queries in production |
| Neo4j CMDB | CI-Incident-Team graph | Questions about dependencies require graph traversal |
| Hybrid RAG | Vector + Cypher combined | Some questions need both semantic + structural retrieval |
| RAGAS-style eval | Faithfulness + relevance scored | Measure before shipping; a score <0.7 needs investigation |
| Conflicting docs | Old VPN policy retrieved alongside new | Tag deprecated docs; add recency weighting |

**Next week:** Agents — the model calls real APIs, creates real tickets, and needs real guardrails.